# Marathon notebook

A **long** notebook (45 cells) for scroll, outline, kernel and
save-performance testing: sections of setup, analysis, narration
and one deliberate error.

* safe to re-run — every cell is pure stdlib


In [ ]:
import math
import random
from statistics import mean, stdev

random.seed(2026)
SENSOR_COUNT = 24
READINGS = 480
print('kernel ready:', SENSOR_COUNT, 'sensors,', READINGS, 'readings each')

kernel ready: 24 sensors, 480 readings each


## 1. Synthetic sensor farm

Each sensor drifts with its own bias and noise level.

In [ ]:
sensors = {
  f'S{i:02d}': {
    'bias' : random.uniform(-2, 2),
    'noise': random.uniform(0.2, 1.4),
  }
  for i in range(SENSOR_COUNT)
}
print(len(sensors), 'sensors initialized')

24 sensors initialized


In [ ]:
rows = []
for t in range(READINGS):
  for name, cfg in sensors.items():
    base = 20 + 6 * math.sin(2 * math.pi * t / READINGS)
    rows.append((t, name, base + cfg['bias'] + random.gauss(0, cfg['noise'])))
print(f'{len(rows)} rows generated')

11520 rows generated


## 2. Per-sensor statistics

Mean and standard deviation per sensor, printed as a compact table.

In [ ]:
table = sorted((name, (m, s)) for name, (m, s) in stats.items())
for name, (m, s) in table:
  print(f'{name}  {m:6.2f}  {s:5.2f}')

sensor   mean   stdev
S00   19.30   0.61
S01   20.60   0.50
S02   20.14   0.91
S03   18.23   1.11
S04   18.15   1.01
S05   18.28   0.53
S06   19.70   1.56
S07   18.50   0.71
S08   20.51   1.73
S09   20.31   0.96
S10   21.91   0.47
S11   21.43   0.81
S12   18.58   0.56
S13   19.23   1.54
S14   18.72   1.21
S15   20.56   0.92
S16   20.19   0.49
S17   18.24   0.69
S18   20.72   1.00
S19   19.26   1.22
S20   19.81   0.82
S21   21.18   1.38
S22   18.98   1.20
S23   20.10   1.63


In [ ]:
# This cell FAILS on purpose — the error output rendering is part of the test.
1 / 0

ZeroDivisionError: division by zero

The `ZeroDivisionError` above is intentional — the notebook viewer
should render the traceback exactly like Jupyter does.

## 3. Rolling windows

A rolling mean with a hand-rolled window (no pandas) over the
aggregate signal.

In [ ]:
aggregate = [mean(r[2] for r in rows if r[0] == t) for t in range(READINGS)]
WINDOW = 12
rolling = [mean(aggregate[max(0, t - WINDOW + 1): t + 1]) for t in range(READINGS)]
print('first 5 rolling:', [round(v, 2) for v in rolling[:5]])

first 5 rolling: [19.87, 19.85, 19.82, 19.8, 19.79]


## 3.1 Harmonic amplitudes

Section 1 of the marathon — re-run live or read the baked output.

In [ ]:
harmonic = [round(sum(math.sin(2 * math.pi * k * t / READINGS) * v for t, v in enumerate(aggregate)) / READINGS, 4) for
            k in (1, 2, 3)]
print(harmonic)

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.2 Quantile buckets

Section 2 of the marathon — re-run live or read the baked output.

In [ ]:
ordered = sorted(aggregate)
for q in (0.1, 0.25, 0.5, 0.75, 0.9):
  print(f'p{int(q * 100):02d}: {ordered[int(q * (len(ordered) - 1))]:.2f}')

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.3 Extrema hunt

Section 3 of the marathon — re-run live or read the baked output.

In [ ]:
print('min', round(min(aggregate), 2), 'max', round(max(aggregate), 2))
print('argmin', aggregate.index(min(aggregate)), 'argmax', aggregate.index(max(aggregate)))

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.4 Resampling

Section 4 of the marathon — re-run live or read the baked output.

In [ ]:
hourly = [mean(aggregate[i: i + 20]) for i in range(0, READINGS, 20)]
print(len(hourly), 'hourly points:', [round(v, 1) for v in hourly])

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.5 Threshold alerts

Section 5 of the marathon — re-run live or read the baked output.

In [ ]:
hot = [t for t, v in enumerate(aggregate) if v > 24]
cold = [t for t, v in enumerate(aggregate) if v < 16]
print('hot ticks:', len(hot), '· cold ticks:', len(cold))

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.6 Smoothing comparison

Section 6 of the marathon — re-run live or read the baked output.

In [ ]:
def boxcar(seq, w):
  return [mean(seq[max(0, i - w + 1): i + 1]) for i in range(len(seq))]


for w in (3, 7, 21):
  sm = boxcar(aggregate, w)
  print(f'w={w:2d}: head {[round(v, 2) for v in sm[:4]]}')

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.7 Cross-sensor pairs

Section 7 of the marathon — re-run live or read the baked output.

In [ ]:
import itertools

pairs = list(itertools.combinations(sorted(sensors)[:6], 2))
print('sampled', len(pairs), 'pairs:', pairs[:3])

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.8 Histogram (ASCII)

Section 8 of the marathon — re-run live or read the baked output.

In [ ]:
bins = [0] * 10
for v in aggregate:
  bins[min(9, int((v - 12) / 1.5))] += 1
for i, n in enumerate(bins):
  print(f'{12 + i * 1.5:4.0f}-{14 + i * 1.5:3.0f} ' + '#' * n)

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.9 Anomaly scan

Section 9 of the marathon — re-run live or read the baked output.

In [ ]:
mu, sigma = mean(aggregate), stdev(aggregate)
anomalies = [(t, round(v, 2)) for t, v in enumerate(aggregate) if abs(v - mu) > 3 * sigma]
print('3-sigma anomalies:', anomalies or 'none')

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 3.10 Long text cell

Section 10 of the marathon — re-run live or read the baked output.

In [ ]:
story = 'the sensors hum a quiet song ' * 40
print(story[:300], '…')

Notes:

- the cell above is deterministic under the fixed seed
- re-running overwrites nothing on disk


## 4. Wrap-up

- 45 cells, one deliberate error, one raw cell
- scroll + outline performance is the point
- Save writes back through the normal .ipynb flow


In [ ]:
print('marathon notebook: cells =', 45, '· rows =', len(rows))

marathon notebook: cells = 45 · rows = 11520


<!-- filler section 44 — padding the notebook to the 45-cell target -->

<!-- filler section 45 — padding the notebook to the 45-cell target -->